# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_08 — Futures Minimum Variance Hedge Ratio

### Research question

How do we estimate, validate, and implement a futures hedge ratio that minimises variance of a spot or portfolio exposure, especially when basis risk and cross-hedging make a naïve 1-for-1 hedge unreliable?

This notebook follows:

```text
04_03_volatility_targeting_and_position_sizing
04_06_var_cvar_and_stress_testing
04_07_beta_weighted_tail_risk_hedging
```

The previous notebook used beta-weighted market hedges. This notebook focuses on **futures hedge ratios**.

It covers:

1. long and short futures hedges;
2. minimum-variance hedge ratio derivation;
3. covariance formula $h^*=\rho\sigma_S/\sigma_F$;
4. OLS hedge-ratio estimation;
5. hedge effectiveness;
6. train/test hedge validation;
7. basis risk;
8. cross-hedging;
9. rolling hedge ratios;
10. downside hedge ratios;
11. multi-futures hedge ratios;
12. contract sizing with multipliers;
13. hedge PnL attribution;
14. stress scenarios;
15. tail-risk before/after hedging;
16. governance checks;
17. saved outputs and manifest.

The key idea:

> The optimal futures hedge is not automatically one contract of futures for one unit of spot exposure. It depends on covariance, volatility, basis risk, contract value, and hedge objective.

## 1. Short hedge versus long hedge

A futures hedge depends on the underlying exposure.

### Short hedge

Used when you own or produce the asset and fear price declines.

Example:

```text
long physical copper exposure + short copper futures
```

Hedged change:

$$
\Delta V_{hedged} = \Delta S - h\Delta F
$$

### Long hedge

Used when you will need to buy the asset and fear price increases.

Example:

```text
future oil purchase + long oil futures
```

Hedged change:

$$
\Delta V_{hedged} = -\Delta S + h\Delta F
$$

This notebook focuses primarily on short hedges for long spot exposure, then shows sign conventions for contract sizing.

## 2. Minimum variance hedge ratio

For a short hedge of a long spot exposure:

$$
R_h = R_S - hR_F
$$

The hedge variance is:

$$
Var(R_h)=Var(R_S-hR_F)
$$

$$
= \sigma_S^2 - 2hCov(R_S,R_F) + h^2\sigma_F^2
$$

Differentiate with respect to $h$:

$$
\begin{aligned}
\frac{d}{dh}Var(R_h) &= -2Cov(R_S,R_F) \\
&\quad + 2h\sigma_F^2
\end{aligned}
$$

Set equal to zero:

$$
h^* = \frac{Cov(R_S,R_F)}{Var(R_F)}
$$

Equivalently:

$$
h^*=\rho_{S,F}\frac{\sigma_S}{\sigma_F}
$$

This is the minimum-variance hedge ratio.

## 3. Hedge effectiveness

A common hedge effectiveness metric:

$$
HE = 1 - \frac{Var(R_{hedged})}{Var(R_{unhedged})}
$$

If $HE=0.75$, the hedge reduces return variance by 75%.

But variance reduction is not the only objective.

A risk manager should also check:

- downside losses;
- VaR and CVaR;
- basis risk;
- turnover;
- contract liquidity;
- margin;
- cash-flow timing;
- stress performance.

## 4. Contract sizing

If the hedge ratio is $h^*$, spot exposure value is $V_S$, futures price is $F$, and contract multiplier is $M$:

$$
N^* = h^* \frac{V_S}{F M}
$$

For a short hedge, the actual futures position is usually:

$$
-N^*
$$

contracts.

For a long hedge, it is:

$$
+N^*
$$

contracts.

This notebook reports signed contract counts explicitly.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class FuturesMVHRConfig:
    n_days: int = 2_200
    train_fraction: float = 0.65
    annualisation: int = 252
    rolling_window: int = 126
    hedge_horizon_days: int = 1
    downside_quantile: float = 0.15
    initial_spot_exposure_usd: float = 5_000_000.0
    transaction_cost_bps_futures: float = 1.0
    contract_multiplier_copper: float = 5.0
    contract_multiplier_oil: float = 1000.0
    contract_multiplier_gold: float = 100.0
    contract_multiplier_index: float = 50.0
    seed: int = 42


config = FuturesMVHRConfig()
config

## 6. Synthetic spot and futures market

We simulate:

### Spot exposures

- copper spot;
- oil spot;
- gold spot;
- equity index spot;
- commodity basket exposure.

### Futures contracts

- copper futures;
- oil futures;
- gold futures;
- equity index futures;
- broad commodity futures.

The simulation includes:

1. correlated spot/futures changes;
2. basis process;
3. contango/backwardation carry;
4. stress events;
5. basis shocks;
6. imperfect cross-hedges.

This allows us to test direct hedges and cross-hedges.

In [ ]:
def simulate_spot_futures_market(config: FuturesMVHRConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2015-01-01", periods=config.n_days)

    spot_names = ["COPPER_SPOT", "OIL_SPOT", "GOLD_SPOT", "EQUITY_SPOT"]
    futures_names = ["COPPER_FUT", "OIL_FUT", "GOLD_FUT", "INDEX_FUT", "COMMODITY_FUT"]

    # Factor returns.
    market = np.zeros(config.n_days)
    commodity = np.zeros(config.n_days)
    rates = np.zeros(config.n_days)
    gold_factor = np.zeros(config.n_days)
    oil_factor = np.zeros(config.n_days)
    copper_factor = np.zeros(config.n_days)
    basis_shock = np.zeros(config.n_days)
    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    transition = np.array([[0.985, 0.015], [0.050, 0.950]])

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_mult = 1.0 if regime[t] == 0 else 2.3

        market[t] = 0.00025 + vol_mult * 0.010 * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)
        commodity[t] = 0.00010 + vol_mult * 0.012 * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)
        rates[t] = 0.00003 + vol_mult * 0.003 * rng.standard_normal()
        gold_factor[t] = 0.00008 + 0.35 * commodity[t] + 0.20 * rates[t] + vol_mult * 0.007 * rng.standard_normal()
        oil_factor[t] = 0.00010 + 0.55 * commodity[t] + 0.25 * market[t] + vol_mult * 0.013 * rng.standard_normal()
        copper_factor[t] = 0.00012 + 0.65 * commodity[t] + 0.35 * market[t] + vol_mult * 0.010 * rng.standard_normal()
        basis_shock[t] = vol_mult * 0.0035 * rng.standard_t(df=7) * np.sqrt((7 - 2) / 7)

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "global_risk_off"
            market[t] -= rng.uniform(0.035, 0.090)
            commodity[t] -= rng.uniform(0.015, 0.070)
            rates[t] += rng.uniform(0.003, 0.020)
            gold_factor[t] += rng.uniform(0.005, 0.035)
            oil_factor[t] -= rng.uniform(0.020, 0.080)
            copper_factor[t] -= rng.uniform(0.020, 0.090)
            basis_shock[t] += rng.normal(0.0, 0.020)
        elif u < 0.014:
            stress_type[t] = "inflation_commodity_spike"
            commodity[t] += rng.uniform(0.030, 0.100)
            rates[t] -= rng.uniform(0.005, 0.030)
            oil_factor[t] += rng.uniform(0.040, 0.120)
            copper_factor[t] += rng.uniform(0.020, 0.070)
            gold_factor[t] += rng.uniform(0.020, 0.060)
            basis_shock[t] += rng.normal(0.0, 0.018)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            commodity[t] -= rng.uniform(0.040, 0.120)
            oil_factor[t] -= rng.uniform(0.050, 0.150)
            copper_factor[t] -= rng.uniform(0.040, 0.120)
            market[t] -= rng.uniform(0.005, 0.035)
            basis_shock[t] += rng.normal(0.0, 0.018)
        elif u < 0.024:
            stress_type[t] = "basis_dislocation"
            basis_shock[t] += rng.normal(0.0, 0.050)

    # Spot log returns.
    spot_returns = pd.DataFrame({
        "COPPER_SPOT": copper_factor + 0.004 * rng.standard_normal(config.n_days),
        "OIL_SPOT": oil_factor + 0.006 * rng.standard_normal(config.n_days),
        "GOLD_SPOT": gold_factor + 0.004 * rng.standard_normal(config.n_days),
        "EQUITY_SPOT": market + 0.003 * rng.standard_normal(config.n_days),
    }, index=dates)

    # Futures returns: related to spot but with basis and carry noise.
    futures_returns = pd.DataFrame(index=dates)
    futures_returns["COPPER_FUT"] = 0.92 * spot_returns["COPPER_SPOT"] + 0.08 * commodity + basis_shock + 0.0025 * rng.standard_normal(config.n_days)
    futures_returns["OIL_FUT"] = 0.90 * spot_returns["OIL_SPOT"] + 0.07 * commodity + 0.80 * basis_shock + 0.0040 * rng.standard_normal(config.n_days)
    futures_returns["GOLD_FUT"] = 0.94 * spot_returns["GOLD_SPOT"] + 0.04 * commodity + 0.50 * basis_shock + 0.0025 * rng.standard_normal(config.n_days)
    futures_returns["INDEX_FUT"] = 0.98 * spot_returns["EQUITY_SPOT"] + 0.15 * basis_shock + 0.0020 * rng.standard_normal(config.n_days)
    futures_returns["COMMODITY_FUT"] = 0.35 * futures_returns["COPPER_FUT"] + 0.35 * futures_returns["OIL_FUT"] + 0.30 * futures_returns["GOLD_FUT"] + 0.003 * rng.standard_normal(config.n_days)

    # Prices from returns.
    start_prices = {
        "COPPER_SPOT": 9000.0,
        "OIL_SPOT": 75.0,
        "GOLD_SPOT": 1900.0,
        "EQUITY_SPOT": 5000.0,
        "COPPER_FUT": 9050.0,
        "OIL_FUT": 76.0,
        "GOLD_FUT": 1905.0,
        "INDEX_FUT": 5010.0,
        "COMMODITY_FUT": 100.0,
    }

    all_returns = pd.concat([spot_returns, futures_returns], axis=1)
    prices = pd.DataFrame(index=dates)

    for col in all_returns.columns:
        prices[col.replace("_SPOT", "_PRICE").replace("_FUT", "_PRICE")] = start_prices[col] * np.exp(np.cumsum(all_returns[col]))

    # Basis = futures price - spot price for direct pairs.
    basis = pd.DataFrame(index=dates)
    basis["COPPER_BASIS"] = prices["COPPER_PRICE"] - prices["COPPER_PRICE"]  # placeholder overwritten below

    # Rename price columns more clearly.
    price_map = {
        "COPPER_PRICE": "COPPER_SPOT_PRICE",
        "OIL_PRICE": "OIL_SPOT_PRICE",
        "GOLD_PRICE": "GOLD_SPOT_PRICE",
        "EQUITY_PRICE": "EQUITY_SPOT_PRICE",
    }

    # The replacement above creates collisions, so build prices directly instead.
    prices = pd.DataFrame(index=dates)
    for col in spot_returns.columns:
        prices[f"{col}_PRICE"] = start_prices[col] * np.exp(np.cumsum(spot_returns[col]))
    for col in futures_returns.columns:
        prices[f"{col}_PRICE"] = start_prices[col] * np.exp(np.cumsum(futures_returns[col]))

    basis = pd.DataFrame(index=dates)
    basis["COPPER_BASIS"] = prices["COPPER_FUT_PRICE"] - prices["COPPER_SPOT_PRICE"]
    basis["OIL_BASIS"] = prices["OIL_FUT_PRICE"] - prices["OIL_SPOT_PRICE"]
    basis["GOLD_BASIS"] = prices["GOLD_FUT_PRICE"] - prices["GOLD_SPOT_PRICE"]
    basis["INDEX_BASIS"] = prices["INDEX_FUT_PRICE"] - prices["EQUITY_SPOT_PRICE"]

    metadata = pd.DataFrame([
        {"contract": "COPPER_FUT", "underlying": "COPPER_SPOT", "multiplier": config.contract_multiplier_copper, "sector": "metals"},
        {"contract": "OIL_FUT", "underlying": "OIL_SPOT", "multiplier": config.contract_multiplier_oil, "sector": "energy"},
        {"contract": "GOLD_FUT", "underlying": "GOLD_SPOT", "multiplier": config.contract_multiplier_gold, "sector": "metals"},
        {"contract": "INDEX_FUT", "underlying": "EQUITY_SPOT", "multiplier": config.contract_multiplier_index, "sector": "equity"},
        {"contract": "COMMODITY_FUT", "underlying": "COMMODITY_BASKET", "multiplier": 1000.0, "sector": "commodity_basket"},
    ])

    market_info = pd.DataFrame({
        "date": dates,
        "regime": regime,
        "stress_type": stress_type,
        "market_factor": market,
        "commodity_factor": commodity,
        "rates_factor": rates,
        "basis_shock": basis_shock,
    })

    all_returns.insert(0, "date", dates)
    prices.insert(0, "date", dates)
    basis.insert(0, "date", dates)

    return all_returns, prices, basis, market_info, metadata


returns_df, prices_df, basis_df, market_info, contract_metadata = simulate_spot_futures_market(config)

returns_df.head(), prices_df.head(), basis_df.head(), contract_metadata

In [ ]:
returns = returns_df.set_index("date")
prices = prices_df.set_index("date")
basis = basis_df.set_index("date")

plt.figure(figsize=(12, 6))
for col in ["COPPER_SPOT", "COPPER_FUT", "OIL_SPOT", "OIL_FUT", "GOLD_SPOT", "GOLD_FUT"]:
    plt.plot(returns.index, (1 + returns[col]).cumprod(), label=col, alpha=0.85)
plt.title("Spot and Futures Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(12, 5))
for col in ["COPPER_BASIS", "OIL_BASIS", "GOLD_BASIS"]:
    plt.plot(basis.index, basis[col], label=col)
plt.title("Synthetic Basis")
plt.xlabel("Date")
plt.ylabel("Futures price minus spot price")
plt.legend()
plt.show()

## 7. Hedge-ratio estimation

For a short hedge:

$$
R_h = R_S - hR_F
$$

The minimum variance hedge ratio is:

$$
h^* = \frac{Cov(R_S,R_F)} {Var(R_F)}
$$

This is also the OLS slope from:

$$
R_S = \alpha + hR_F + \epsilon
$$

where $R_S$ is the spot exposure return and $R_F$ is the futures return.

In [ ]:
def estimate_mvhr(spot_return, futures_return):
    data = pd.concat([spot_return.rename("spot"), futures_return.rename("futures")], axis=1).dropna()
    cov = data["spot"].cov(data["futures"])
    var_f = data["futures"].var(ddof=1)
    h = cov / var_f if var_f > 0 else np.nan

    corr = data["spot"].corr(data["futures"])
    vol_spot = data["spot"].std(ddof=1)
    vol_fut = data["futures"].std(ddof=1)
    h_corr = corr * vol_spot / vol_fut if vol_fut > 0 else np.nan

    alpha = data["spot"].mean() - h * data["futures"].mean()
    residual = data["spot"] - (alpha + h * data["futures"])
    r2 = 1 - (residual @ residual) / ((data["spot"] - data["spot"].mean()) @ (data["spot"] - data["spot"].mean()))

    return {
        "hedge_ratio": float(h),
        "hedge_ratio_corr_formula": float(h_corr),
        "correlation": float(corr),
        "spot_vol_ann": float(vol_spot * np.sqrt(config.annualisation)),
        "futures_vol_ann": float(vol_fut * np.sqrt(config.annualisation)),
        "ols_alpha_daily": float(alpha),
        "ols_r2": float(r2),
        "n_obs": int(len(data)),
    }


direct_pairs = {
    "COPPER_SPOT": "COPPER_FUT",
    "OIL_SPOT": "OIL_FUT",
    "GOLD_SPOT": "GOLD_FUT",
    "EQUITY_SPOT": "INDEX_FUT",
}

mvhr_rows = []

for spot, fut in direct_pairs.items():
    res = estimate_mvhr(returns[spot], returns[fut])
    mvhr_rows.append({"spot": spot, "futures": fut, **res})

mvhr_table_full = pd.DataFrame(mvhr_rows)
mvhr_table_full

## 8. Train/test split

Hedge ratios should be estimated on historical data and validated out of sample.

We split:

1. train period: estimate hedge ratios;
2. test period: evaluate hedge effectiveness.

In [ ]:
split_idx = int(len(returns) * config.train_fraction)

train_returns = returns.iloc[:split_idx].copy()
test_returns = returns.iloc[split_idx:].copy()

split_metadata = {
    "n_total_days": int(len(returns)),
    "n_train_days": int(len(train_returns)),
    "n_test_days": int(len(test_returns)),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
}

pd.Series(split_metadata)

In [ ]:
mvhr_train_rows = []

for spot, fut in direct_pairs.items():
    res = estimate_mvhr(train_returns[spot], train_returns[fut])
    mvhr_train_rows.append({"spot": spot, "futures": fut, **res})

mvhr_train = pd.DataFrame(mvhr_train_rows)
mvhr_train

## 9. Hedge effectiveness

Out-of-sample hedge return for a short hedge:

$$
R_{hedged,t}=R_{S,t}-hR_{F,t}
$$

Hedge effectiveness:

$$
HE = 1 - \frac{Var(R_{hedged})} {Var(R_S)}
$$

We compare:

1. unhedged;
2. naïve $h=1$;
3. train-estimated minimum-variance hedge ratio.

In [ ]:
def hedge_return_short(spot_return, futures_return, h):
    return spot_return - h * futures_return


def hedge_effectiveness(unhedged_return, hedged_return):
    var_unhedged = unhedged_return.var(ddof=1)
    var_hedged = hedged_return.var(ddof=1)
    return 1 - var_hedged / var_unhedged if var_unhedged > 0 else np.nan


def summarise_hedge_performance(spot_return, futures_return, h, label):
    hedged = hedge_return_short(spot_return, futures_return, h)

    return {
        "hedge": label,
        "h": h,
        "mean_ann": float(hedged.mean() * config.annualisation),
        "vol_ann": float(hedged.std(ddof=1) * np.sqrt(config.annualisation)),
        "worst_day": float(hedged.min()),
        "best_day": float(hedged.max()),
        "hedge_effectiveness": float(hedge_effectiveness(spot_return, hedged)),
        "corr_hedged_with_spot": float(hedged.corr(spot_return)),
    }


hedge_perf_rows = []
hedge_returns = {}

for _, row in mvhr_train.iterrows():
    spot = row["spot"]
    fut = row["futures"]
    h_star = row["hedge_ratio"]

    spot_test = test_returns[spot]
    fut_test = test_returns[fut]

    for h, label in [(0.0, "unhedged"), (1.0, "naive_1_to_1"), (h_star, "minimum_variance")]:
        perf = summarise_hedge_performance(spot_test, fut_test, h, label)
        hedge_perf_rows.append({"spot": spot, "futures": fut, **perf})
        hedge_returns[(spot, label)] = hedge_return_short(spot_test, fut_test, h)

hedge_performance = pd.DataFrame(hedge_perf_rows)
hedge_performance

In [ ]:
plt.figure(figsize=(12, 6))
for spot in ["COPPER_SPOT", "OIL_SPOT", "GOLD_SPOT"]:
    sub = hedge_performance[(hedge_performance["spot"] == spot)]
    plt.plot(sub["hedge"], sub["hedge_effectiveness"], marker="o", label=spot)
plt.axhline(0, linestyle="--")
plt.title("Out-of-Sample Hedge Effectiveness")
plt.xlabel("Hedge")
plt.ylabel("Variance reduction")
plt.legend()
plt.show()

## 10. Visualising hedged returns

We plot cumulative value of:

- unhedged spot exposure;
- naïve hedge;
- minimum-variance hedge.

This shows whether variance reduction is achieved at the cost of drift or basis exposure.

In [ ]:
spot_to_plot = "COPPER_SPOT"

plt.figure(figsize=(12, 6))
for label in ["unhedged", "naive_1_to_1", "minimum_variance"]:
    series = hedge_returns[(spot_to_plot, label)]
    plt.plot(series.index, (1 + series).cumprod(), label=label)
plt.title(f"Hedged Return Growth: {spot_to_plot}")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 11. Cross-hedging

Sometimes the exact futures contract is unavailable or illiquid.

Example:

```text
copper exposure hedged with broad commodity futures
```

Cross-hedge ratio:

$$
h^* = \frac{Cov(R_{spot}, R_{cross\ futures})} {Var(R_{cross\ futures})}
$$

Cross-hedges generally have lower correlation and lower hedge effectiveness.

In [ ]:
cross_pairs = [
    ("COPPER_SPOT", "COMMODITY_FUT"),
    ("OIL_SPOT", "COMMODITY_FUT"),
    ("GOLD_SPOT", "COMMODITY_FUT"),
    ("COPPER_SPOT", "OIL_FUT"),
    ("OIL_SPOT", "COPPER_FUT"),
    ("EQUITY_SPOT", "COMMODITY_FUT"),
]

cross_rows = []

for spot, fut in cross_pairs:
    train_res = estimate_mvhr(train_returns[spot], train_returns[fut])
    h = train_res["hedge_ratio"]
    test_hedged = hedge_return_short(test_returns[spot], test_returns[fut], h)

    cross_rows.append({
        "spot": spot,
        "cross_futures": fut,
        "train_hedge_ratio": h,
        "train_corr": train_res["correlation"],
        "train_r2": train_res["ols_r2"],
        "test_hedge_effectiveness": hedge_effectiveness(test_returns[spot], test_hedged),
        "test_hedged_vol_ann": test_hedged.std(ddof=1) * np.sqrt(config.annualisation),
        "test_unhedged_vol_ann": test_returns[spot].std(ddof=1) * np.sqrt(config.annualisation),
    })

cross_hedge_report = pd.DataFrame(cross_rows).sort_values("test_hedge_effectiveness", ascending=False)
cross_hedge_report

## 12. Basis risk

Basis:

$$
Basis_t = F_t - S_t
$$

For a perfect hedge with the same asset and expiry, basis risk may still exist because:

- futures and spot do not move one-for-one;
- delivery location differs;
- quality differs;
- expiry mismatch;
- liquidity shocks;
- funding/carry changes;
- contract rolls.

The hedged return can be interpreted as basis exposure.

In [ ]:
basis_summary = basis.agg(["mean", "std", "min", "max"]).T
basis_summary = basis_summary.rename(columns={
    "mean": "basis_mean",
    "std": "basis_std",
    "min": "basis_min",
    "max": "basis_max",
})

basis_summary

In [ ]:
plt.figure(figsize=(12, 6))
for col in basis.columns:
    plt.plot(basis.index, basis[col], label=col)
plt.title("Basis Through Time")
plt.xlabel("Date")
plt.ylabel("Basis")
plt.legend()
plt.show()

## 13. Rolling hedge ratios

Hedge ratios can change over time.

Rolling estimator:

$$
h_t^* = \frac{Cov_t(R_S,R_F)} {Var_t(R_F)}
$$

A rolling hedge adapts but creates turnover and estimation noise.

In [ ]:
def rolling_mvhr(spot_return, futures_return, window):
    cov = spot_return.rolling(window).cov(futures_return)
    var = futures_return.rolling(window).var()
    h = cov / var
    return h.replace([np.inf, -np.inf], np.nan)


rolling_h = pd.DataFrame(index=returns.index)

for spot, fut in direct_pairs.items():
    rolling_h[f"{spot}_vs_{fut}"] = rolling_mvhr(returns[spot], returns[fut], config.rolling_window)

rolling_h.tail()

In [ ]:
plt.figure(figsize=(12, 6))
for col in rolling_h.columns:
    plt.plot(rolling_h.index, rolling_h[col], label=col)
plt.axhline(1.0, linestyle="--", label="h=1")
plt.title("Rolling Minimum-Variance Hedge Ratios")
plt.xlabel("Date")
plt.ylabel("Hedge ratio")
plt.legend(ncol=2)
plt.show()

## 14. Rolling hedge backtest

Use yesterday's rolling hedge ratio to hedge today's return:

$$
R_{hedged,t}=R_{S,t}-h_{t-1}R_{F,t}
$$

Transaction costs are proportional to hedge-ratio changes.

In [ ]:
def rolling_hedge_backtest(spot_return, futures_return, rolling_h, transaction_cost_bps):
    h_lag = rolling_h.shift(1).fillna(0.0)

    gross = spot_return - h_lag * futures_return
    turnover = h_lag.diff().abs().fillna(h_lag.abs())
    cost = turnover * transaction_cost_bps / 10000.0
    net = gross - cost

    return pd.DataFrame({
        "spot_return": spot_return,
        "futures_return": futures_return,
        "hedge_ratio": h_lag,
        "gross_hedged_return": gross,
        "turnover": turnover,
        "transaction_cost": cost,
        "net_hedged_return": net
    })


rolling_backtests = {}
rolling_perf_rows = []

for spot, fut in direct_pairs.items():
    h_series = rolling_h[f"{spot}_vs_{fut}"]
    bt = rolling_hedge_backtest(returns[spot], returns[fut], h_series, config.transaction_cost_bps_futures)
    rolling_backtests[spot] = bt

    # Evaluate only after enough rolling history.
    eval_bt = bt.iloc[config.rolling_window:].dropna()
    unhedged = eval_bt["spot_return"]
    hedged = eval_bt["net_hedged_return"]

    rolling_perf_rows.append({
        "spot": spot,
        "futures": fut,
        "rolling_hedge_effectiveness_after_cost": hedge_effectiveness(unhedged, hedged),
        "unhedged_vol_ann": unhedged.std(ddof=1) * np.sqrt(config.annualisation),
        "hedged_vol_ann": hedged.std(ddof=1) * np.sqrt(config.annualisation),
        "mean_turnover": eval_bt["turnover"].mean(),
        "annualised_cost_drag": eval_bt["transaction_cost"].mean() * config.annualisation,
        "mean_hedge_ratio": eval_bt["hedge_ratio"].mean(),
        "std_hedge_ratio": eval_bt["hedge_ratio"].std(ddof=1),
    })

rolling_hedge_performance = pd.DataFrame(rolling_perf_rows)
rolling_hedge_performance

## 15. Downside hedge ratio

A risk manager may care more about hedge performance during bad spot days.

Downside MVHR is estimated only when spot return is in the lower tail:

$$
h_{down}^* = \frac{Cov(R_S,R_F \mid R_S \le q_\alpha)} {Var(R_F \mid R_S \le q_\alpha)}
$$

This can differ from normal MVHR.

In [ ]:
def downside_mvhr(spot_return, futures_return, quantile):
    threshold = spot_return.quantile(quantile)
    mask = spot_return <= threshold
    return estimate_mvhr(spot_return.loc[mask], futures_return.loc[mask])


downside_rows = []

for spot, fut in direct_pairs.items():
    normal = estimate_mvhr(train_returns[spot], train_returns[fut])
    downside = downside_mvhr(train_returns[spot], train_returns[fut], config.downside_quantile)

    h_down = downside["hedge_ratio"]
    test_hedged_down = hedge_return_short(test_returns[spot], test_returns[fut], h_down)
    test_hedged_normal = hedge_return_short(test_returns[spot], test_returns[fut], normal["hedge_ratio"])

    tail_threshold = test_returns[spot].quantile(config.downside_quantile)
    tail_mask = test_returns[spot] <= tail_threshold

    downside_rows.append({
        "spot": spot,
        "futures": fut,
        "normal_mvhr": normal["hedge_ratio"],
        "downside_mvhr": h_down,
        "normal_corr": normal["correlation"],
        "downside_corr": downside["correlation"],
        "test_normal_HE": hedge_effectiveness(test_returns[spot], test_hedged_normal),
        "test_downside_HE": hedge_effectiveness(test_returns[spot], test_hedged_down),
        "tail_mean_unhedged": test_returns[spot].loc[tail_mask].mean(),
        "tail_mean_normal_hedged": test_hedged_normal.loc[tail_mask].mean(),
        "tail_mean_downside_hedged": test_hedged_down.loc[tail_mask].mean(),
    })

downside_hedge_report = pd.DataFrame(downside_rows)
downside_hedge_report

## 16. Multi-futures hedge ratio

For multiple hedge instruments $F$, the variance-minimising hedge vector is:

$$
h^* = \Sigma_{FF}^{-1} \Sigma_{FS}
$$

where:

- $\Sigma_{FF}$ is covariance matrix of futures returns;
- $\Sigma_{FS}$ is covariance vector between futures returns and spot return.

For a short hedge:

$$
R_h = R_S - h^\top R_F
$$

This is multivariate regression of spot returns on futures returns.

In [ ]:
def estimate_multi_futures_hedge(spot_return, futures_returns):
    data = pd.concat([spot_return.rename("spot"), futures_returns], axis=1).dropna()
    y = data["spot"].to_numpy()
    X = data[futures_returns.columns].to_numpy()

    X_design = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y

    alpha = beta[0]
    h = pd.Series(beta[1:], index=futures_returns.columns)

    fitted = alpha + X @ h.to_numpy()
    residual = y - fitted
    r2 = 1 - (residual @ residual) / ((y - y.mean()) @ (y - y.mean()))

    return {
        "alpha": alpha,
        "hedge_ratios": h,
        "r2": r2,
        "n_obs": len(data),
    }


multi_hedge_instruments = ["COPPER_FUT", "OIL_FUT", "GOLD_FUT", "COMMODITY_FUT"]

multi_rows = []
multi_hedged_returns = {}

for spot in ["COPPER_SPOT", "OIL_SPOT", "GOLD_SPOT"]:
    fit = estimate_multi_futures_hedge(train_returns[spot], train_returns[multi_hedge_instruments])
    h = fit["hedge_ratios"]

    test_hedged = test_returns[spot] - (test_returns[multi_hedge_instruments] @ h)
    multi_hedged_returns[spot] = test_hedged

    row = {
        "spot": spot,
        "train_r2": fit["r2"],
        "test_HE": hedge_effectiveness(test_returns[spot], test_hedged),
        "test_hedged_vol_ann": test_hedged.std(ddof=1) * np.sqrt(config.annualisation),
        "test_unhedged_vol_ann": test_returns[spot].std(ddof=1) * np.sqrt(config.annualisation),
    }

    for fut in multi_hedge_instruments:
        row[f"h_{fut}"] = h[fut]

    multi_rows.append(row)

multi_futures_report = pd.DataFrame(multi_rows).sort_values("test_HE", ascending=False)
multi_futures_report

## 17. Hedge horizon sensitivity

Hedge ratios can depend on return horizon.

For $k$-day returns:

$$
R^{(k)}_t = \prod_{j=0}^{k-1}(1+R_{t-j})-1
$$

We estimate MVHR at multiple horizons:

- 1 day;
- 5 days;
- 21 days.

A hedge ratio useful for daily risk may not be ideal for monthly exposure.

In [ ]:
def horizon_return(series, horizon):
    return (1 + series).rolling(horizon).apply(lambda x: np.prod(x) - 1, raw=True)


horizon_rows = []

for horizon in [1, 5, 21]:
    for spot, fut in direct_pairs.items():
        spot_h = horizon_return(returns[spot], horizon).dropna()
        fut_h = horizon_return(returns[fut], horizon).dropna()
        joined = pd.concat([spot_h.rename("spot"), fut_h.rename("futures")], axis=1).dropna()

        res = estimate_mvhr(joined["spot"], joined["futures"])
        horizon_rows.append({
            "horizon_days": horizon,
            "spot": spot,
            "futures": fut,
            "hedge_ratio": res["hedge_ratio"],
            "correlation": res["correlation"],
            "ols_r2": res["ols_r2"],
            "spot_vol_ann_scaled": joined["spot"].std(ddof=1) * np.sqrt(config.annualisation / horizon),
            "futures_vol_ann_scaled": joined["futures"].std(ddof=1) * np.sqrt(config.annualisation / horizon),
        })

horizon_sensitivity = pd.DataFrame(horizon_rows)
horizon_sensitivity

In [ ]:
plt.figure(figsize=(12, 6))
for spot in direct_pairs.keys():
    sub = horizon_sensitivity[horizon_sensitivity["spot"] == spot]
    plt.plot(sub["horizon_days"], sub["hedge_ratio"], marker="o", label=spot)
plt.axhline(1.0, linestyle="--")
plt.title("Hedge Ratio by Return Horizon")
plt.xlabel("Horizon days")
plt.ylabel("MVHR")
plt.legend()
plt.show()

## 18. Contract sizing

For a short hedge:

$$
N^* = -h^*\frac{V_S}{F M}
$$

where:

- $V_S$: spot exposure value;
- $F$: futures price;
- $M$: contract multiplier;
- negative sign indicates short futures.

We compute contract counts using last available futures price.

In [ ]:
def contract_count_for_short_hedge(spot_exposure_value, hedge_ratio, futures_price, multiplier):
    contracts = -hedge_ratio * spot_exposure_value / (futures_price * multiplier)
    return contracts


contract_rows = []

for _, row in mvhr_train.iterrows():
    spot = row["spot"]
    fut = row["futures"]
    h = row["hedge_ratio"]

    multiplier = contract_metadata.set_index("contract").loc[fut, "multiplier"]
    futures_price = prices[f"{fut}_PRICE"].iloc[-1]

    contracts = contract_count_for_short_hedge(
        config.initial_spot_exposure_usd,
        h,
        futures_price,
        multiplier
    )

    contract_rows.append({
        "spot": spot,
        "futures": fut,
        "hedge_ratio": h,
        "spot_exposure_usd": config.initial_spot_exposure_usd,
        "futures_price": futures_price,
        "multiplier": multiplier,
        "signed_contracts_short_hedge": contracts,
        "rounded_contracts": int(np.round(contracts)),
        "contract_notional_usd": futures_price * multiplier,
    })

contract_sizing = pd.DataFrame(contract_rows)
contract_sizing

## 19. Contract rounding error

Futures contracts are discrete.

After rounding contracts, the implemented hedge ratio becomes:

$$
\hat h = -\frac{N_{rounded} F M}{V_S}
$$

Rounding creates hedge slippage, especially for small exposures or large contracts.

In [ ]:
rounding_rows = []

for _, row in contract_sizing.iterrows():
    implemented_h = -row["rounded_contracts"] * row["futures_price"] * row["multiplier"] / row["spot_exposure_usd"]

    rounding_rows.append({
        "spot": row["spot"],
        "futures": row["futures"],
        "target_h": row["hedge_ratio"],
        "rounded_contracts": row["rounded_contracts"],
        "implemented_h": implemented_h,
        "hedge_ratio_error": implemented_h - row["hedge_ratio"],
        "abs_error": abs(implemented_h - row["hedge_ratio"]),
    })

rounding_report = pd.DataFrame(rounding_rows)
rounding_report

## 20. Hedge PnL attribution

For a short hedge:

$$
R_{hedged}=R_S-hR_F
$$

The hedged return decomposes into:

1. spot return;
2. futures hedge PnL;
3. transaction costs;
4. residual basis risk.

We compute attribution for each direct hedge.

In [ ]:
def hedge_pnl_attribution(spot, fut, h, returns):
    spot_r = returns[spot]
    fut_r = returns[fut]
    futures_pnl = -h * fut_r
    hedged = spot_r + futures_pnl

    out = pd.DataFrame({
        "spot": spot,
        "futures": fut,
        "spot_return": spot_r,
        "futures_hedge_pnl": futures_pnl,
        "hedged_return": hedged,
        "residual_basis_like_return": hedged
    })

    return out


attr_frames = []

for _, row in mvhr_train.iterrows():
    attr = hedge_pnl_attribution(row["spot"], row["futures"], row["hedge_ratio"], test_returns)
    attr["hedge_ratio"] = row["hedge_ratio"]
    attr_frames.append(attr.reset_index().rename(columns={"index": "date"}))

hedge_attribution = pd.concat(attr_frames, ignore_index=True)
hedge_attribution.head()

In [ ]:
attr_summary = (
    hedge_attribution
    .groupby(["spot", "futures"])
    .agg(
        mean_spot_ann=("spot_return", lambda x: x.mean() * config.annualisation),
        mean_futures_hedge_pnl_ann=("futures_hedge_pnl", lambda x: x.mean() * config.annualisation),
        mean_hedged_ann=("hedged_return", lambda x: x.mean() * config.annualisation),
        vol_spot_ann=("spot_return", lambda x: x.std(ddof=1) * np.sqrt(config.annualisation)),
        vol_hedged_ann=("hedged_return", lambda x: x.std(ddof=1) * np.sqrt(config.annualisation)),
    )
    .reset_index()
)

attr_summary

## 21. VaR/CVaR before and after hedging

Variance reduction is useful, but hedges should also be judged on tail risk.

We compare historical VaR and CVaR for:

- unhedged spot;
- naïve hedge;
- minimum-variance hedge;
- rolling hedge;
- downside hedge;
- multi-futures hedge where available.

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


tail_rows = []

for spot, fut in direct_pairs.items():
    h_normal = mvhr_train.loc[mvhr_train["spot"] == spot, "hedge_ratio"].iloc[0]
    h_down = downside_hedge_report.loc[downside_hedge_report["spot"] == spot, "downside_mvhr"].iloc[0]

    strategy_series = {
        "unhedged": test_returns[spot],
        "naive_1_to_1": hedge_return_short(test_returns[spot], test_returns[fut], 1.0),
        "minimum_variance": hedge_return_short(test_returns[spot], test_returns[fut], h_normal),
        "downside_mvhr": hedge_return_short(test_returns[spot], test_returns[fut], h_down),
    }

    if spot in rolling_backtests:
        rb = rolling_backtests[spot].reindex(test_returns.index)
        strategy_series["rolling_mvhr_after_cost"] = rb["net_hedged_return"]

    if spot in multi_hedged_returns:
        strategy_series["multi_futures_mvhr"] = multi_hedged_returns[spot]

    for strategy, series in strategy_series.items():
        losses = -series.dropna()
        for alpha in [0.95, 0.99]:
            var, cvar = historical_var_cvar(losses, alpha)
            tail_rows.append({
                "spot": spot,
                "strategy": strategy,
                "alpha": alpha,
                "VaR": var,
                "CVaR": cvar,
            })

tail_risk_hedges = pd.DataFrame(tail_rows)
tail_risk_hedges.head()

In [ ]:
spot_to_plot = "OIL_SPOT"
sub = tail_risk_hedges[(tail_risk_hedges["spot"] == spot_to_plot) & (tail_risk_hedges["alpha"] == 0.99)].sort_values("CVaR")

plt.figure(figsize=(10, 6))
plt.barh(sub["strategy"], sub["CVaR"])
plt.title(f"99% CVaR by Hedge Strategy: {spot_to_plot}")
plt.xlabel("CVaR loss")
plt.ylabel("Strategy")
plt.show()

## 22. Stress testing hedge ratios

Hedge ratios estimated in normal periods may fail in stress.

We evaluate hedge returns on synthetic stress days:

- global risk-off;
- inflation commodity spike;
- commodity crash;
- basis dislocation.

Basis dislocation is especially dangerous because it breaks spot-futures co-movement.

In [ ]:
stress_info = market_info.set_index("date").reindex(returns.index)
stress_mask = stress_info["stress_type"] != "normal"

stress_rows = []

for spot, fut in direct_pairs.items():
    h = mvhr_train.loc[mvhr_train["spot"] == spot, "hedge_ratio"].iloc[0]
    hedged = hedge_return_short(returns[spot], returns[fut], h)
    naive = hedge_return_short(returns[spot], returns[fut], 1.0)

    for stress_type, idx in stress_info.groupby("stress_type").groups.items():
        if stress_type == "normal":
            continue

        idx = pd.Index(idx)
        idx = idx.intersection(returns.index)

        if len(idx) == 0:
            continue

        stress_rows.append({
            "spot": spot,
            "futures": fut,
            "stress_type": stress_type,
            "n_days": len(idx),
            "mean_unhedged": returns.loc[idx, spot].mean(),
            "mean_naive_hedged": naive.loc[idx].mean(),
            "mean_mvhr_hedged": hedged.loc[idx].mean(),
            "worst_unhedged": returns.loc[idx, spot].min(),
            "worst_naive_hedged": naive.loc[idx].min(),
            "worst_mvhr_hedged": hedged.loc[idx].min(),
        })

stress_hedge_report = pd.DataFrame(stress_rows)
stress_hedge_report.head()

In [ ]:
plot_stress = stress_hedge_report[stress_hedge_report["spot"] == "COPPER_SPOT"]

plt.figure(figsize=(12, 6))
plt.plot(plot_stress["stress_type"], -plot_stress["worst_unhedged"], marker="o", label="unhedged loss")
plt.plot(plot_stress["stress_type"], -plot_stress["worst_naive_hedged"], marker="x", label="naive hedge loss")
plt.plot(plot_stress["stress_type"], -plot_stress["worst_mvhr_hedged"], marker="s", label="MVHR hedge loss")
plt.xticks(rotation=45, ha="right")
plt.title("Worst Stress-Day Loss: COPPER_SPOT")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 23. Hedge effectiveness dashboard

We collect the main diagnostics:

1. train hedge ratio;
2. train $R^2$;
3. out-of-sample hedge effectiveness;
4. rolling hedge effectiveness;
5. downside hedge effect;
6. tail-risk effect;
7. basis volatility;
8. contract sizing.

In [ ]:
dashboard_rows = []

for spot, fut in direct_pairs.items():
    base = mvhr_train[mvhr_train["spot"] == spot].iloc[0]
    test_perf = hedge_performance[
        (hedge_performance["spot"] == spot)
        & (hedge_performance["hedge"] == "minimum_variance")
    ].iloc[0]

    roll_row = rolling_hedge_performance[rolling_hedge_performance["spot"] == spot].iloc[0]
    down_row = downside_hedge_report[downside_hedge_report["spot"] == spot].iloc[0]
    contract_row = contract_sizing[contract_sizing["spot"] == spot].iloc[0]

    tail_unhedged = tail_risk_hedges[
        (tail_risk_hedges["spot"] == spot)
        & (tail_risk_hedges["strategy"] == "unhedged")
        & (tail_risk_hedges["alpha"] == 0.99)
    ]["CVaR"].iloc[0]

    tail_mvhr = tail_risk_hedges[
        (tail_risk_hedges["spot"] == spot)
        & (tail_risk_hedges["strategy"] == "minimum_variance")
        & (tail_risk_hedges["alpha"] == 0.99)
    ]["CVaR"].iloc[0]

    basis_col = {
        "COPPER_SPOT": "COPPER_BASIS",
        "OIL_SPOT": "OIL_BASIS",
        "GOLD_SPOT": "GOLD_BASIS",
        "EQUITY_SPOT": "INDEX_BASIS",
    }[spot]

    dashboard_rows.append({
        "spot": spot,
        "futures": fut,
        "train_mvhr": base["hedge_ratio"],
        "train_corr": base["correlation"],
        "train_r2": base["ols_r2"],
        "test_HE_mvhr": test_perf["hedge_effectiveness"],
        "test_HE_rolling_after_cost": roll_row["rolling_hedge_effectiveness_after_cost"],
        "normal_mvhr": down_row["normal_mvhr"],
        "downside_mvhr": down_row["downside_mvhr"],
        "cvar99_unhedged": tail_unhedged,
        "cvar99_mvhr": tail_mvhr,
        "cvar99_reduction": 1 - tail_mvhr / tail_unhedged if tail_unhedged != 0 else np.nan,
        "basis_std": basis[basis_col].std(ddof=1),
        "rounded_contracts": contract_row["rounded_contracts"],
        "rounding_abs_error": rounding_report[rounding_report["spot"] == spot]["abs_error"].iloc[0],
    })

hedge_dashboard = pd.DataFrame(dashboard_rows).sort_values("test_HE_mvhr", ascending=False)
hedge_dashboard

## 24. Governance flags

A hedge should be reviewed if:

1. train correlation is weak;
2. out-of-sample hedge effectiveness is low;
3. CVaR reduction is poor;
4. rolling hedge ratio is unstable;
5. contract rounding error is large;
6. basis volatility is high.

These are process controls, not universal thresholds.

In [ ]:
governance_flags = hedge_dashboard.copy()

governance_flags["flag_low_train_corr"] = governance_flags["train_corr"].abs() < 0.70
governance_flags["flag_low_test_HE"] = governance_flags["test_HE_mvhr"] < 0.40
governance_flags["flag_low_cvar_reduction"] = governance_flags["cvar99_reduction"] < 0.20
governance_flags["flag_large_rounding_error"] = governance_flags["rounding_abs_error"] > 0.05
governance_flags["review_required"] = governance_flags[
    [
        "flag_low_train_corr",
        "flag_low_test_HE",
        "flag_low_cvar_reduction",
        "flag_large_rounding_error",
    ]
].any(axis=1)

governance_flags

## 25. Practical checklist for futures MVHR

Before implementing a futures hedge:

1. **Identify exposure**  
   Long spot exposure needs short futures; future purchase risk needs long futures.

2. **Choose hedge instrument**  
   Direct future if possible; cross-hedge if necessary.

3. **Estimate MVHR**  
   Use covariance or OLS slope.

4. **Validate out of sample**  
   Never assume in-sample $R^2$ guarantees future hedge effectiveness.

5. **Check basis risk**  
   Basis shocks can dominate hedge error.

6. **Check horizon**  
   Daily hedge ratios may differ from monthly hedge ratios.

7. **Check downside behaviour**  
   Normal hedge ratio may not protect tail risk.

8. **Convert to contracts**  
   Use futures price and contract multiplier.

9. **Account for rounding**  
   Contracts are discrete.

10. **Monitor rolling hedge ratio**  
   Hedge ratios drift.

11. **Track costs and margin**  
   Futures hedges create cash-flow and collateral requirements.

12. **Stress test**  
   Especially basis dislocation and liquidity events.

## 26. Saving outputs

The notebook saves:

1. synthetic spot/futures returns;
2. synthetic prices;
3. basis table;
4. market/stress metadata;
5. contract metadata;
6. full-sample MVHR table;
7. train MVHR table;
8. hedge performance report;
9. cross-hedge report;
10. basis summary;
11. rolling hedge ratios;
12. rolling hedge performance;
13. downside hedge report;
14. multi-futures hedge report;
15. horizon sensitivity;
16. contract sizing;
17. rounding report;
18. hedge attribution;
19. tail-risk table;
20. stress hedge report;
21. hedge dashboard;
22. governance flags;
23. manifest.

In [ ]:
output_dir = Path("data/processed/futures_minimum_variance_hedge_ratio_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "spot_futures_returns.csv"
prices_path = output_dir / "spot_futures_prices.csv"
basis_path = output_dir / "basis.csv"
market_info_path = output_dir / "market_info.csv"
contract_metadata_path = output_dir / "contract_metadata.csv"
mvhr_full_path = output_dir / "mvhr_full_sample.csv"
mvhr_train_path = output_dir / "mvhr_train.csv"
hedge_performance_path = output_dir / "hedge_performance.csv"
cross_hedge_path = output_dir / "cross_hedge_report.csv"
basis_summary_path = output_dir / "basis_summary.csv"
rolling_h_path = output_dir / "rolling_hedge_ratios.csv"
rolling_perf_path = output_dir / "rolling_hedge_performance.csv"
downside_path = output_dir / "downside_hedge_report.csv"
multi_futures_path = output_dir / "multi_futures_report.csv"
horizon_sensitivity_path = output_dir / "horizon_sensitivity.csv"
contract_sizing_path = output_dir / "contract_sizing.csv"
rounding_report_path = output_dir / "rounding_report.csv"
hedge_attribution_path = output_dir / "hedge_attribution.csv"
attr_summary_path = output_dir / "hedge_attribution_summary.csv"
tail_risk_path = output_dir / "tail_risk_hedges.csv"
stress_hedge_path = output_dir / "stress_hedge_report.csv"
hedge_dashboard_path = output_dir / "hedge_dashboard.csv"
governance_flags_path = output_dir / "governance_flags.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
prices_df.to_csv(prices_path, index=False)
basis_df.to_csv(basis_path, index=False)
market_info.to_csv(market_info_path, index=False)
contract_metadata.to_csv(contract_metadata_path, index=False)
mvhr_table_full.to_csv(mvhr_full_path, index=False)
mvhr_train.to_csv(mvhr_train_path, index=False)
hedge_performance.to_csv(hedge_performance_path, index=False)
cross_hedge_report.to_csv(cross_hedge_path, index=False)
basis_summary.to_csv(basis_summary_path)
rolling_h.to_csv(rolling_h_path)
rolling_hedge_performance.to_csv(rolling_perf_path, index=False)
downside_hedge_report.to_csv(downside_path, index=False)
multi_futures_report.to_csv(multi_futures_path, index=False)
horizon_sensitivity.to_csv(horizon_sensitivity_path, index=False)
contract_sizing.to_csv(contract_sizing_path, index=False)
rounding_report.to_csv(rounding_report_path, index=False)
hedge_attribution.to_csv(hedge_attribution_path, index=False)
attr_summary.to_csv(attr_summary_path, index=False)
tail_risk_hedges.to_csv(tail_risk_path, index=False)
stress_hedge_report.to_csv(stress_hedge_path, index=False)
hedge_dashboard.to_csv(hedge_dashboard_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)

manifest = {
    "dataset_name": "futures_minimum_variance_hedge_ratio_outputs",
    "schema_version": "futures_minimum_variance_hedge_ratio_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "direct_pairs": direct_pairs,
    "split_metadata": split_metadata,
    "hedge_dashboard": hedge_dashboard.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "notes": [
        "Minimum-variance hedge ratio is estimated as Cov(spot, futures) / Var(futures).",
        "OLS slope and covariance formula are shown to match.",
        "Short hedge convention uses R_hedged = R_spot - h R_futures.",
        "Direct hedges, cross-hedges, rolling hedges, downside hedges, and multi-futures hedges are compared.",
        "Contract counts use signed short-hedge convention: N = -h * exposure / (F * multiplier).",
        "Basis risk, hedge horizon sensitivity, CVaR impact, stress performance, and rounding error are reported.",
        "This notebook is synthetic and educational; production futures hedging requires contract expiry, liquidity, margin, tax, and exchange-specific rules."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, mvhr_train_path, hedge_dashboard_path, governance_flags_path, manifest_path

## 27. Limitations

### 27.1 Synthetic data

The notebook uses synthetic spot and futures returns.

Real hedging requires actual spot exposure data, futures settlements, roll calendars, liquidity, and contract specifications.

### 27.2 Returns versus price changes

This notebook estimates hedge ratios using returns.

Some commodity hedges are estimated using price changes instead, depending on exposure definition.

### 27.3 Static hedge assumptions

Static MVHR can become stale.

Rolling hedges adapt but add turnover and estimation noise.

### 27.4 Basis risk

Basis risk can dominate hedge error, especially during dislocations.

### 27.5 Simplified transaction costs

Transaction costs are fixed bps.

Real costs depend on spread, market impact, slippage, exchange fees, and liquidity.

### 27.6 No margin accounting

Futures hedges require margin and daily variation settlement.

This notebook excludes collateral and cash-flow constraints.

### 27.7 Contract expiry ignored

Production futures hedging must handle expiries, rolls, delivery rules, and changing contract liquidity.

### 27.8 Cross-hedge instability

Cross-hedges can break down when correlations change.

## 28. Research frontier and extensions

Important extensions include:

1. **Price-change hedge ratios**  
   Estimate $\Delta S$ versus $\Delta F$ in price units.

2. **Cointegration and error-correction hedges**  
   Use spot/futures long-run relationships.

3. **State-space hedge ratios**  
   Kalman-filter hedge ratio dynamics.

4. **Regime-switching hedge ratios**  
   Different hedge ratios in calm and stress regimes.

5. **CVaR-minimising hedge ratios**  
   Optimise tail loss directly rather than variance.

6. **Liquidity-adjusted hedge ratios**  
   Penalise hedge instruments with high trading costs.

7. **Contract roll-aware hedging**  
   Use active contract and roll schedule.

8. **Margin-aware hedging**  
   Include variation margin and collateral liquidity.

9. **Multi-commodity cross hedging**  
   Hedge physical commodity exposure using a basket of futures.

10. **Chinese futures extension**  
   Copper, rebar, iron ore, PTA, soybean meal, night sessions, price limits, and exchange-specific contract multipliers.

## 29. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_09_risk_parity_and_equal_risk_contribution**  
   Allocate by risk contribution.

2. **04_10_portfolio_constraints_margin_and_leverage**  
   Add margin and futures constraints.

3. **05_03_transaction_costs_slippage_latency**  
   Model execution costs for rolling futures hedges.

4. **05_04_walk_forward_hedge_validation**  
   Formal out-of-sample hedge evaluation.

5. **06_05_futures_contract_roll_engine**  
   Production futures infrastructure.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use hedge ratios in a futures portfolio system.

## 30. Summary

This notebook implemented futures minimum-variance hedge ratios.

It showed how to:

1. simulate spot and futures returns with basis risk;
2. derive the minimum-variance hedge ratio;
3. estimate hedge ratios using covariance and OLS;
4. split train/test periods;
5. compare unhedged, naïve, and MVHR hedges;
6. calculate hedge effectiveness;
7. analyse basis risk;
8. estimate cross-hedges;
9. compute rolling hedge ratios;
10. backtest rolling hedges with transaction costs;
11. estimate downside hedge ratios;
12. estimate multi-futures hedge vectors;
13. test hedge-ratio horizon sensitivity;
14. convert hedge ratios into futures contract counts;
15. measure rounding error;
16. attribute hedge PnL;
17. compare VaR/CVaR before and after hedging;
18. stress test hedge performance;
19. create governance flags.

The key computational takeaway:

> The minimum-variance hedge ratio is a regression slope, but implementation requires timing, contracts, costs, and validation.

The key financial takeaway:

> A futures hedge reduces risk only to the extent that futures returns reliably offset the actual exposure; basis risk is the enemy.

## 31. Further reading

- Hull, *Options, Futures, and Other Derivatives.*
- Ederington on futures hedging effectiveness.
- Working on hedging and basis risk.
- Commodity futures hedging literature.
- Risk management texts on cross-hedging, minimum-variance hedge ratios, and hedge effectiveness.